In [0]:
%run ../00_common/calc_ctable

In [0]:
task_id = dbutils.widgets.get("task_id")
print(f"task_id: {task_id}")

# master_consumer_table_name = f"{get_env_config('golden_consumer_master_database')}.t_master_consumer"
master_derived_consumer_l1_table_name = f"{get_env_config('golden_consumer_master_database')}.t_derived_consumer_l1"
master_derived_consumer_l2_table_name = f"{get_env_config('golden_consumer_master_database')}.t_derived_consumer_l2"
master_derived_consumer_l3_table_name = f"{get_env_config('golden_consumer_master_database')}.t_derived_consumer_l3"
dataset_cbr_table_name = f"{get_env_config('golden_consumer_combine_database')}.t_cbr_dataset"
dataset_cbrldrjart_table_name = f"{get_env_config('golden_consumer_combine_database')}.t_cbrdrjart_dataset"
dataset_nonpii_cbr_table_name = f"{get_env_config('golden_consumer_combine_database')}.t_cbr_withoutpii_dataset"

timestamp_format = "yyyy-MM-dd'T'HH:mm:ss"

# 正则表达式 (用于最后移除空数组字段)
pattern = r'(,"AffiliateCode":\[\]|,"MarketCode":\[\]|,"DivisionCode":\[\]|,"BrandCode":\[\]|,"TerminalId":\[\]|,"PersonnelCode":\[\]|"SourceTimestamp":\[\]|,"EformRegFlag":\[\]|,"RegPersonnelSourceSystem":\{\}|,"NameGenderCode":\[\]|,"GenderCode":\[\]|,"CivilStatusCode":\[\]|,"ConsumerCountryCode_ISO3":\[\]|,"SpokenLanguageCode":\[\]|,"WrittenLanguageCode":\[\]|,"EstimatedBirthYear":\[\]|,"RetailerHieararchyCode":\[\]|,"TouchPointCode":\[\]|,"LocalFirstName2":\[\]|,"LocalMiddleName2":\[\]|,"LocalLastName2":\[\]|,"LocalFullName2":\[\]|,"IdentityNum":\[\]|,"PassportNum":\[\]|,"SocialSecurityNum":\[\]|,"Company":\[\]|,"Department":\[\]|,"JobTitle":\[\]|,"YearlyIncome":\[\]|,"SkinTypeCode":\[\]|,"BrandCode":\[\]|,"SkinType":\{\}|,"CurrencyCode":\[\]|,"EthnicityCode":\[\]|,"MarketCode":\[\]|,"Ethnicity":\{\}|,"PreferredTouchPointCodeSourceSystem":\{\}|,"AssignedPersonnelSourceSystem":\{\}|,"HobbyList":\[\]|,"GeneralOptInFlag":\[\]|,"GeneralOptInDate":\[\]|,"SkinConcernsList":\[\]|,"HairTypeList":\[\]|,"MakeUpConcernList":\[\]|"AgeFrom":\[\]|,"AgeTo":\[\]|,"AgeRange":\{\}|,"HairConcernsList":\[\]|,"Nationality":\[\]|,"OptInSourceSystemCode":\[\])'

In [0]:
# SourceSystem Schema
source_system_list_schema = StructType([
    StructField("SourceSystem", ArrayType(StructType([
        StructField("@Code", StringType(), True),
        StructField("SourceTimestamp", StringType(), True),
        StructField("AffiliateCode", StringType(), True),
        StructField("MarketCode", StringType(), True),
        StructField("DivisionCode", StringType(), True),
        StructField("BrandCode", StringType(), True),
        StructField("ConsumerId", StringType(), True),
        StructField("TerminalId", StringType(), True),
        StructField("DeleteFlag", BooleanType(), True)
    ])), True)
])

# HobbyList Schema
hobby_list_schema = StructType([
    StructField("Hobby", ArrayType(StructType([
        StructField("HobbyDescription", StringType(), True)
    ])), True)
])

# EMediaList Schema
emedia_list_schema = StructType([
    StructField("EMedia", ArrayType(StructType([
        StructField("@TypeCode", StringType(), True),
        StructField("SourceTimestamp", StringType(), True),
        StructField("Address", StringType(), True),
        StructField("DataQualityCode", StringType(), True),
        StructField("DataQualityDescription", StringType(), True)
    ])), True)
])

# EMediaList NonPii Schema
emedia_list_nonpii_schema = StructType([
    StructField("EMedia", ArrayType(StructType([
        StructField("@TypeCode", StringType(), True),
        StructField("SourceTimestamp", StringType(), True),
        StructField("DataQualityCode", StringType(), True),
        StructField("DataQualityDescription", StringType(), True)
    ])), True)
])

# PhoneList Schema
phone_list_schema = StructType([
    StructField("Phone", ArrayType(StructType([
        StructField("@TypeCode", StringType(), True),
        StructField("SourceTimestamp", StringType(), True),
        StructField("CountryCode_ISO3", StringType(), True),
        StructField("PhoneCountryCode", StringType(), True),
        StructField("RegionalCode", StringType(), True),
        StructField("Extension", StringType(), True),
        StructField("PhoneNumber", StringType(), True),
        StructField("RegPhoneTypeCode", StringType(), True),
        StructField("Registrar", StringType(), True),
        StructField("DataQualityCode", StringType(), True),
        StructField("DataQualityDescription", StringType(), True)
    ])), True)
])

# PhoneList NonPii Schema
phone_list_nonpii_schema = StructType([
    StructField("Phone", ArrayType(StructType([
        StructField("@TypeCode", StringType(), True),
        StructField("SourceTimestamp", StringType(), True),
        StructField("DataQualityCode", StringType(), True),
        StructField("DataQualityDescription", StringType(), True)
    ])), True)
])

# AddressList Schema
address_list_schema = StructType([
    StructField("Address", ArrayType(StructType([
        StructField("@TypeCode", StringType(), True),
        StructField("SourceTimestamp", StringType(), True),
        StructField("Address1", StringType(), True),
        StructField("Address2", StringType(), True),
        StructField("Address3", StringType(), True),
        StructField("Address4", StringType(), True),
        StructField("Address5", StringType(), True),
        StructField("FlatNo", StringType(), True),
        StructField("Floor", StringType(), True),
        StructField("Block", StringType(), True),
        StructField("Phase", StringType(), True),
        StructField("Building", StringType(), True),
        StructField("StreetNo", StringType(), True),
        StructField("StreetNoSuffix", StringType(), True),
        StructField("Alley", StringType(), True),
        StructField("StreetName", StringType(), True),
        StructField("StreetName2", StringType(), True),
        StructField("Estate", StringType(), True),
        StructField("LaneNo", StringType(), True),
        StructField("LaneName", StringType(), True),
        StructField("Sector", StringType(), True),
        StructField("POBox", StringType(), True),
        StructField("PostalCode", StringType(), True),
        StructField("SubCityCode", StringType(), True),
        StructField("SubCityCodeDescription_en", StringType(), True),
        StructField("SubCityCodeDescription_local", StringType(), True),
        StructField("CityCode", StringType(), True),
        StructField("CityDescription_en", StringType(), True),
        StructField("CityDescription_local", StringType(), True),
        StructField("Province", StringType(), True),
        StructField("ProvinceCode", StringType(), True),
        StructField("ProvinceDescription_en", StringType(), True),
        StructField("ProvinceDescription_local", StringType(), True),
        StructField("AdminArea", StringType(), True),
        StructField("CountryCode_ISO3", StringType(), True),
        StructField("NUTSCode", StringType(), True),
        StructField("Longitude", StringType(), True),
        StructField("Latitude", StringType(), True),
        StructField("GeocodeResolution", StringType(), True),
        StructField("GeographicalSpokenLanguage", StringType(), True),
        StructField("DataQualityCode", StringType(), True),
        StructField("DataQualityDescription", StringType(), True)
    ])), True)
])

# AddressList NonPii Schema
address_list_nonpii_schema = StructType([
    StructField("Address", ArrayType(StructType([
        StructField("@TypeCode", StringType(), True),
        StructField("SourceTimestamp", StringType(), True),
        StructField("CityDescription_local", StringType(), True),
        StructField("ProvinceDescription_local", StringType(), True),
        StructField("DataQualityCode", StringType(), True),
        StructField("DataQualityDescription", StringType(), True)
    ])), True)
])

# OptInList Schema
opt_in_list_schema = StructType([
    StructField("OptIn", ArrayType(StructType([
        StructField("OptInTimestamp", StringType(), True),
        StructField("CommunicationChannelCode", StringType(), True),
        StructField("OptInFlag", BooleanType(), True),
        StructField("OptInSourceSystemCode", StringType(), True)
    ])), True)
])

# CrossBrandOptInList Schema
cross_brand_opt_in_list_schema = StructType([
    StructField("CrossBrandOptIn", ArrayType(StructType([
        StructField("OptInTimestamp", StringType(), True),
        StructField("OptInFlag", BooleanType(), True)
    ])), True)
])

# CustomAttributeList Schema
custom_attribute_list_schema = StructType([
    StructField("CustomAttribute", ArrayType(StructType([
        StructField("@Name", StringType(), True),
        StructField("@Value", StringType(), True)
    ])), True)
])

# AuxiliaryAttributeList Schema
auxiliary_attribute_list_schema = StructType([
    StructField("AuxiliaryAttribute", ArrayType(StructType([
        StructField("Code", StringType(), True),
        StructField("Description", StringType(), True),
        StructField("MultiValue", BooleanType(), True),
        StructField("Value", StringType(), True),
        StructField("Active", BooleanType(), True)
    ])), True)
])

# HairTypeList Schema
hair_type_list_schema = StructType([
    StructField("HairType", ArrayType(StringType()), True)
])

# MakeUpConcernList Schema
make_up_concern_list_schema = StructType([
    StructField("MakeUpConcerns", ArrayType(StringType()), True)
])

# HairConcernsList Schema
hair_concerns_list_schema = StructType([
    StructField("HairConcerns", ArrayType(StringType()), True)
])

# SkinConcernsList Schema
skin_concerns_list_schema = StructType([
    StructField("SkinConcerns", ArrayType(StringType()), True)
])

# TermsAndConditionList Schema
terms_and_condition_list_schema = StructType([
    StructField("TermsAndCondition", ArrayType(StructType([
        StructField("Code", StringType(), True),
        StructField("Description", StringType(), True),
        StructField("Version", StringType(), True),
        StructField("AcceptedDate", StringType(), True)
    ])), True)
])

# CustomerGroupList Schema
customer_group_list_schema = StructType([
    StructField("CustomerGroup", ArrayType(StringType()), True)
])

# RemarkList Schema
remark_list_schema = StructType([
    StructField("Remark", ArrayType(StructType([
        StructField("RemarkCode", StringType(), True),
        StructField("RemarksDate", StringType(), True),
        StructField("Remarks", StringType(), True)
    ])), True)
])

# NoteList Schema
note_list_schema = StructType([
    StructField("Note", ArrayType(StructType([
        StructField("SeqNum", IntegerType(), True),
        StructField("Type", StringType(), True),
        StructField("Location", StringType(), True),
        StructField("Note", StringType(), True),
        StructField("CreateDate", StringType(), True),
        StructField("CreateBy", StringType(), True),
        StructField("UpdateDate", StringType(), True),
        StructField("UpdateBy", StringType(), True)
    ])), True)
])

# ProgramList Schema
program_list_schema = StructType([
    StructField("Program", ArrayType(StructType([
        StructField("ApplicationTouchPointCode", StringType(), True),
        StructField("ConsumerGroup", StringType(), True),
        StructField("ProgramTypeCode", StringType(), True),
        StructField("ProgramTypeDescription", StringType(), True),
        StructField("ProgramLevelCode", StringType(), True),
        StructField("ProgramLevelDescription", StringType(), True),
        StructField("ProgramSystemIDCode", StringType(), True),
        StructField("ProgramSystemIDDescription", StringType(), True),
        StructField("MembershipNum", StringType(), True),
        StructField("CardNum", StringType(), True),
        StructField("StartTimestamp", StringType(), True),
        StructField("EndTimestamp", StringType(), True),
        StructField("PointsAcquired", DecimalType(19, 4), True),
        StructField("PointsRedeemed", DecimalType(19, 4), True),
        StructField("InitialQuota", DecimalType(19, 4), True),
        StructField("AvailableQuota", DecimalType(19, 4), True)
    ])), True)
])

In [0]:
def build_attributes_json_struct():
    """构建Attributes JSON结构"""
    return F.struct(
        F.date_format(F.col("RecordTimeStamp"), timestamp_format).alias("RecordTimestamp"),
        F.date_format(F.col("scon_sourcetimestamp"), timestamp_format).alias("SourceTimestamp"),
        F.col("scon_mrkt_code").alias("MarketCode"),
        F.col("scon_brnd_code").alias("BrandCode"),
        F.col("tcpm_distributionchannelcode").alias("DistributionChannelCode"),
        F.lit(None).cast(StringType()).alias("TouchPointCode")  # 没有对应字段
    )

def build_personal_data_json_struct():
    """构建PersonalData JSON结构"""
    return F.struct(
        # RegDate
        F.date_format(F.col("scon_reg_dt"), timestamp_format).alias("RegDate"),
        
        # RegTouchPointSourceSystem
        F.struct(
            F.date_format(F.col("scon_reg_sourcetimestamp"), timestamp_format).alias("SourceTimestamp"),
            F.col("scon_aff_code").alias("AffiliateCode"),
            F.col("scon_mrkt_code").alias("MarketCode"),
            F.col("scon_dvsn_code").alias("DivisionCode"),
            F.col("scon_brnd_code").alias("BrandCode"),
            F.col("scon_registration_toch_code").alias("TouchPointCode")
        ).alias("RegTouchPointSourceSystem"),
        
        # LastUpdateTouchPointSourceSystem
        F.struct(
            F.date_format(F.col("scon_last_update_timestamp"), timestamp_format).alias("SourceTimestamp"),
            F.col("scon_aff_code").alias("AffiliateCode"),
            F.col("scon_mrkt_code").alias("MarketCode"),
            F.col("scon_dvsn_code").alias("DivisionCode"),
            F.col("scon_brnd_code").alias("BrandCode"),
            F.col("scon_last_update_toch_code").alias("TouchPointCode")
        ).alias("LastUpdateTouchPointSourceSystem"),
        
        # RegPersonnelSourceSystem
        F.struct(
            F.col("scon_prsn_srcs_code").alias("@Code"),
            F.date_format(F.col("scon_prsn_sourcetimestamp"), timestamp_format).alias("SourceTimestamp"),
            F.col("scon_prsn_aff_code").alias("AffiliateCode"),
            F.col("scon_prsn_mrkt_code").alias("MarketCode"),
            F.col("scon_prsn_dvsn_code").alias("DivisionCode"),
            F.col("scon_prsn_brnd_code").alias("BrandCode"),
            F.col("scon_registration_prsn_code").alias("PersonnelCode")
        ).alias("RegPersonnelSourceSystem"),
        
        F.lit(None).cast(StringType()).alias("NameGenderCode"),  # 没有对应字段
        F.col("scon_gndr_code").alias("GenderCode"),
        F.col("scon_cvls_code").alias("CivilStatusCode"),
        F.col("scon_cntr_isoalpha3code").alias("ConsumerCountryCode_ISO3"),
        F.col("scon_slng_code").alias("SpokenLanguageCode"),
        F.col("scon_wlng_code").alias("WrittenLanguageCode"),
        
        # Personal Info
        F.col("scon_birthday").alias("BirthDay"),
        F.col("scon_birthmonth").alias("BirthMonth"),
        F.col("scon_birthyear").alias("BirthYear"),
        F.lit(None).cast(StringType()).alias("EstimatedBirthYear"),  # 没有对应字段
        F.when(F.col("namefilledflag") == 1, True).otherwise(False).alias("NameFilledFlag"),
        F.col("scon_salutation").alias("Salutation"),
        F.col("scon_localfirstname").alias("LocalFirstName"),
        F.col("scon_localmiddlename").alias("LocalMiddleName"),
        F.col("scon_locallastname").alias("LocalLastName"),
        F.col("scon_localfullname").alias("LocalFullName"),
        F.col("scon_localfirstname2").alias("LocalFirstName2"),
        F.col("scon_localmiddlename2").alias("LocalMiddleName2"),
        F.col("scon_locallastname2").alias("LocalLastName2"),
        F.col("scon_localfullname2").alias("LocalFullName2"),
        F.col("scon_englishfirstname").alias("EnglishFirstName"),
        F.col("scon_englishmiddlename").alias("EnglishMiddleName"),
        F.col("scon_englishlastname").alias("EnglishLastName"),
        F.col("scon_englishfullname").alias("EnglishFullName"),
        F.col("scon_identitynum").alias("IdentityNum"),
        F.col("scon_passportnum").alias("PassportNum"),
        F.col("scon_socialsecuritynum").alias("SocialSecurityNum"),
        F.col("scon_company").alias("Company"),
        F.col("scon_department").alias("Department"),
        F.col("scon_jobtitle").alias("JobTitle"),
        F.col("scon_yearlyincome").alias("YearlyIncome"),
        
        # SkinType
        F.struct(
            F.col("scon_sknt_brnd_code").alias("BrandCode"),
            F.col("scon_sknt_code").alias("SkinTypeCode"),
        ).alias("SkinType"),
        
        F.col("scon_curr_code").alias("CurrencyCode"),

        # Ethnicity
        F.struct(
            F.col("scon_ethn_mrkt_code").alias("MarketCode"),
            F.col("scon_ethn_code").alias("EthnicityCode")
        ).alias("Ethnicity"),
        
        F.col("scon_clas_code").alias("ConsumerClassCode"),
        
        # PreferredTouchPointCodeSourceSystem
        F.struct(
            F.date_format(F.col("scon_pre_toch_sourcetimestamp"), timestamp_format).alias("SourceTimestamp"),
            F.col("scon_pre_toch_aff_code").alias("AffiliateCode"),
            F.col("scon_pre_toch_mrkt_code").alias("MarketCode"),
            F.col("scon_pre_toch_dvsn_code").alias("DivisionCode"),
            F.col("scon_pre_toch_brnd_code").alias("BrandCode"),
            F.col("scon_preferred_toch_code").alias("TouchPointCode")
        ).alias("PreferredTouchPointCodeSourceSystem"),
        
        # AssignedPersonnelSourceSystem
        F.struct(
            F.col("scon_assn_prsn_srcs_code").alias("@Code"),
            F.date_format(F.col("scon_assn_prsn_sourcetimestamp"), timestamp_format).alias("SourceTimestamp"),
            F.col("scon_assn_prsn_aff_code").alias("AffiliateCode"),
            F.col("scon_assn_prsn_mrkt_code").alias("MarketCode"),
            F.col("scon_assn_prsn_dvsn_code").alias("DivisionCode"),
            F.col("scon_assn_prsn_brnd_code").alias("BrandCode"),
            F.col("scon_assigned_prsn_code").alias("PersonnelCode")
        ).alias("AssignedPersonnelSourceSystem"),

        F.from_json(F.col("HobbyJSON"), hobby_list_schema).alias("HobbyList"),
        F.lit(None).cast(StringType()).alias("GeneralOptInFlag"),  # 没有对应字段
        F.lit(None).cast(StringType()).alias("GeneralOptInDate"),  # 没有对应字段
        F.lit(False).alias("DoNotContact"),
        F.from_json(F.col("SkinConcernsJSON"), skin_concerns_list_schema).alias("SkinConcernsList"),
        F.from_json(F.col("HairTypeJSON"), hair_type_list_schema).alias("HairTypeList"),
        F.from_json(F.col("MakeUpConcernsJSON"), make_up_concern_list_schema).alias("MakeUpConcernList"),
        
        # AgeRange
        F.struct(
            F.col("scon_agefrom").alias("AgeFrom"),
            F.col("scon_ageto").alias("AgeTo")
        ).alias("AgeRange"),
        
        F.from_json(F.col("HairConcernsJSON"), hair_concerns_list_schema).alias("HairConcernsList"),
        F.col("scon_nationality").alias("Nationality"),
        F.date_format(F.col("scon_firstpurchasedate"), timestamp_format).alias("FirstPurchaseDate")
    )

def build_nonpii_personal_data_json_struct():
    """构建PersonalData NonPii JSON结构"""
    return F.struct(
        # RegDate
        F.date_format(F.col("scon_reg_dt"), timestamp_format).alias("RegDate"),
        
        # RegTouchPointSourceSystem
        F.struct(
            F.date_format(F.col("scon_reg_sourcetimestamp"), timestamp_format).alias("SourceTimestamp"),
            F.col("scon_aff_code").alias("AffiliateCode"),
            F.col("scon_mrkt_code").alias("MarketCode"),
            F.col("scon_dvsn_code").alias("DivisionCode"),
            F.col("scon_brnd_code").alias("BrandCode"),
            F.col("scon_registration_toch_code").alias("TouchPointCode")
        ).alias("RegTouchPointSourceSystem"),
        
        # LastUpdateTouchPointSourceSystem
        F.struct(
            F.date_format(F.col("scon_last_update_timestamp"), timestamp_format).alias("SourceTimestamp"),
            F.col("scon_aff_code").alias("AffiliateCode"),
            F.col("scon_mrkt_code").alias("MarketCode"),
            F.col("scon_dvsn_code").alias("DivisionCode"),
            F.col("scon_brnd_code").alias("BrandCode"),
            F.col("scon_last_update_toch_code").alias("TouchPointCode")
        ).alias("LastUpdateTouchPointSourceSystem"),
        
        # RegPersonnelSourceSystem
        F.struct(
            F.col("scon_prsn_srcs_code").alias("@Code"),
            F.date_format(F.col("scon_prsn_sourcetimestamp"), timestamp_format).alias("SourceTimestamp"),
            F.col("scon_prsn_aff_code").alias("AffiliateCode"),
            F.col("scon_prsn_mrkt_code").alias("MarketCode"),
            F.col("scon_prsn_dvsn_code").alias("DivisionCode"),
            F.col("scon_prsn_brnd_code").alias("BrandCode"),
            F.col("scon_registration_prsn_code").alias("PersonnelCode")
        ).alias("RegPersonnelSourceSystem"),
        
        F.col("scon_gndr_code").alias("GenderCode"),
        F.col("scon_cntr_isoalpha3code").alias("ConsumerCountryCode_ISO3"),
        F.col("scon_slng_code").alias("SpokenLanguageCode"),
        F.col("scon_wlng_code").alias("WrittenLanguageCode"),
        
        # Personal Info
        F.col("scon_birthmonth").alias("BirthMonth"),
        F.col("scon_birthyear").alias("BirthYear"),
        F.when(F.col("namefilledflag") == 1, True).otherwise(False).alias("NameFilledFlag"),
        F.col("scon_salutation").alias("Salutation"),
        
        # SkinType
        F.struct(
            F.col("scon_sknt_brnd_code").alias("BrandCode"),
            F.col("scon_sknt_code").alias("SkinTypeCode"),
        ).alias("SkinType"),

        # Ethnicity
        F.struct(
            F.col("scon_ethn_mrkt_code").alias("MarketCode"),
            F.col("scon_ethn_code").alias("EthnicityCode")
        ).alias("Ethnicity"),
        
        F.col("scon_clas_code").alias("ConsumerClassCode"),
        
        # PreferredTouchPointCodeSourceSystem
        F.struct(
            F.date_format(F.col("scon_pre_toch_sourcetimestamp"), timestamp_format).alias("SourceTimestamp"),
            F.col("scon_pre_toch_aff_code").alias("AffiliateCode"),
            F.col("scon_pre_toch_mrkt_code").alias("MarketCode"),
            F.col("scon_pre_toch_dvsn_code").alias("DivisionCode"),
            F.col("scon_pre_toch_brnd_code").alias("BrandCode"),
            F.col("scon_preferred_toch_code").alias("TouchPointCode")
        ).alias("PreferredTouchPointCodeSourceSystem"),
        
        # AssignedPersonnelSourceSystem
        F.struct(
            F.col("scon_assn_prsn_srcs_code").alias("@Code"),
            F.date_format(F.col("scon_assn_prsn_sourcetimestamp"), timestamp_format).alias("SourceTimestamp"),
            F.col("scon_assn_prsn_aff_code").alias("AffiliateCode"),
            F.col("scon_assn_prsn_mrkt_code").alias("MarketCode"),
            F.col("scon_assn_prsn_dvsn_code").alias("DivisionCode"),
            F.col("scon_assn_prsn_brnd_code").alias("BrandCode"),
            F.col("scon_assigned_prsn_code").alias("PersonnelCode")
        ).alias("AssignedPersonnelSourceSystem"),

        F.lit(None).cast(StringType()).alias("GeneralOptInFlag"),  # 没有对应字段
        F.lit(None).cast(StringType()).alias("GeneralOptInDate"),  # 没有对应字段
        F.lit(False).alias("DoNotContact"),
        F.from_json(F.col("SkinConcernsJSON"), skin_concerns_list_schema).alias("SkinConcernsList"),
        F.from_json(F.col("HairTypeJSON"), hair_type_list_schema).alias("HairTypeList"),
        F.from_json(F.col("MakeUpConcernsJSON"), make_up_concern_list_schema).alias("MakeUpConcernList"),
        
        # AgeRange
        F.struct(
            F.col("scon_agefrom").alias("AgeFrom"),
            F.col("scon_ageto").alias("AgeTo")
        ).alias("AgeRange"),
        
        F.from_json(F.col("HairConcernsJSON"), hair_concerns_list_schema).alias("HairConcernsList"),
        F.col("scon_nationality").alias("Nationality"),
        F.date_format(F.col("scon_firstpurchasedate"), timestamp_format).alias("FirstPurchaseDate")
    )

def build_contact_information_json():
    """构建ContactInformation JSON结构"""
    return F.struct(
        F.from_json(F.col("EMediaJSON"), emedia_list_schema).alias("EMediaList"),
        F.from_json(F.col("PhoneJSON"), phone_list_schema).alias("PhoneList"),
        F.from_json(F.col("AddressJSON"), address_list_schema).alias("AddressList")
    )

def build_nonpii_contact_information_json():
    """构建NonPII ContactInformation JSON结构"""
    return F.struct(
        F.from_json(F.col("EMediaJSON"), emedia_list_nonpii_schema).alias("EMediaList"),
        F.from_json(F.col("PhoneJSON"), phone_list_nonpii_schema).alias("PhoneList"),
        F.from_json(F.col("AddressJSON"), address_list_nonpii_schema).alias("AddressList")
    )

# def build_dbr_json():
#     """构建DerivedBestRecord JSON结构"""
#     return F.struct(
#         F.col("scvlevel").alias("@Level"),
#         F.from_json(F.col("SourceSystemJSON"), source_system_list_schema).alias("SourceSystemList"),
#         # 构建Attributes
#         build_attributes_json_struct().alias("Attributes"),
#         # 构建PersonalData
#         build_personal_data_json_struct().alias("PersonalData"),
#         # 构建ContactInformation
#         build_contact_information_json().alias("ContactInformation"),
#         # 构建NonPii ContactInformation
#         build_nonpii_contact_information_json().alias("ContactInformation_NonPii"),
#         F.from_json(F.col("OptInJSON"), opt_in_list_schema).alias("OptInList"),
#         F.from_json(F.col("CrossBrandOptInJSON"), cross_brand_opt_in_list_schema).alias("CrossBrandOptInList"),
#         F.from_json(F.col("AuxiliaryAttributesJSON"), auxiliary_attribute_list_schema).alias("AuxiliaryAttributeList"),
#         F.from_json(F.col("TermsJSON"), terms_and_condition_list_schema).alias("TermsAndConditionList"),
#         F.from_json(F.col("CustomAttributesJSON"), custom_attribute_list_schema).alias("CustomAttributeList"),
#         F.from_json(F.col("ConsumerGroupJSON"), customer_group_list_schema).alias("CustomerGroupList"),
#         F.from_json(F.col("RemarkJSON"), remark_list_schema).alias("RemarkList"),
#         F.from_json(F.col("NotesJSON"), note_list_schema).alias("NoteList")
#     )

def build_cbr_attributes_json_struct():
    """构建CBR Attributes JSON结构"""
    return F.struct(
        F.date_format(F.col("RecordTimeStamp"), timestamp_format).alias("RecordTimestamp"),
        F.date_format(F.col("scon_sourcetimestamp"), timestamp_format).alias("SourceTimestamp"),
        F.lit("csrcdpapc").alias("CSRInstanceCode"),
        F.lit("CSR CDP — APAC").alias("CSRInstanceDescription"),
        F.col("consumermdmkey").alias("UniversalKey"),
        F.lit("cnsmtlndmdm").alias("RecognitionServiceCode")
    )

def build_cbr_personal_data_json_struct():
    """构建CBR PersonalData JSON结构"""
    return F.struct(
        # RegDate
        F.date_format(F.col("scon_reg_dt"), timestamp_format).alias("RegDate"),
        
        # RegTouchPointSourceSystem
        F.struct(
            F.date_format(F.col("scon_reg_sourcetimestamp"), timestamp_format).alias("SourceTimestamp"),
            F.col("scon_aff_code").alias("AffiliateCode"),
            F.col("scon_mrkt_code").alias("MarketCode"),
            F.col("scon_dvsn_code").alias("DivisionCode"),
            F.col("scon_brnd_code").alias("BrandCode"),
            F.col("scon_registration_toch_code").alias("TouchPointCode")
        ).alias("RegTouchPointSourceSystem"),
        
        # LastUpdateTouchPointSourceSystem
        F.struct(
            F.date_format(F.col("scon_sourcetimestamp"), timestamp_format).alias("SourceTimestamp"),
            F.col("scon_aff_code").alias("AffiliateCode"),
            F.col("scon_mrkt_code").alias("MarketCode"),
            F.col("scon_dvsn_code").alias("DivisionCode"),
            F.col("scon_brnd_code").alias("BrandCode"),
            F.col("scon_lastupdate_toch_code").alias("TouchPointCode")
        ).alias("LastUpdateTouchPointSourceSystem"),
        
        # RegPersonnelSourceSystem
        F.struct(
            F.col("scon_prs_sourcesystemcode").alias("@Code"),
            F.date_format(F.col("scon_prs_sourcetimestamp"), timestamp_format).alias("SourceTimestamp"),
            F.col("scon_prs_aff_code").alias("AffiliateCode"),
            F.col("scon_prs_mrkt_code").alias("MarketCode"),
            F.col("scon_prs_dvsn_code").alias("DivisionCode"),
            F.col("scon_prs_brnd_code").alias("BrandCode"),
            F.col("scon_registration_prsn_code").alias("PersonnelCode")
        ).alias("RegPersonnelSourceSystem"),
        
        F.col("scon_gndr_code").alias("GenderCode"),
        F.col("scon_cvls_code").alias("CivilStatusCode"),
        F.col("scon_cntr_isoalpha3code").alias("ConsumerCountryCode_ISO3"),
        F.col("scon_slng_code").alias("SpokenLanguageCode"),
        F.col("scon_wlng_code").alias("WrittenLanguageCode"),
        
        # Personal Info
        F.col("scon_birthday").alias("BirthDay"),
        F.col("scon_birthmonth").alias("BirthMonth"),
        F.col("scon_birthyear").alias("BirthYear"),
        F.when(F.col("namefilledflag") == 1, True).otherwise(False).alias("NameFilledFlag")
    )

def build_nonpii_cbr_personal_data_json_struct():
    """构建CBR PersonalData NonPii JSON结构"""
    return F.struct(
        # RegDate
        F.date_format(F.col("scon_reg_dt"), timestamp_format).alias("RegDate"),
        
        # RegTouchPointSourceSystem
        F.struct(
            F.date_format(F.col("scon_reg_sourcetimestamp"), timestamp_format).alias("SourceTimestamp"),
            F.col("scon_aff_code").alias("AffiliateCode"),
            F.col("scon_mrkt_code").alias("MarketCode"),
            F.col("scon_dvsn_code").alias("DivisionCode"),
            F.col("scon_brnd_code").alias("BrandCode"),
            F.col("scon_registration_toch_code").alias("TouchPointCode")
        ).alias("RegTouchPointSourceSystem"),
        
        # LastUpdateTouchPointSourceSystem
        F.struct(
            F.date_format(F.col("scon_sourcetimestamp"), timestamp_format).alias("SourceTimestamp"),
            F.col("scon_aff_code").alias("AffiliateCode"),
            F.col("scon_mrkt_code").alias("MarketCode"),
            F.col("scon_dvsn_code").alias("DivisionCode"),
            F.col("scon_brnd_code").alias("BrandCode"),
            F.col("scon_lastupdate_toch_code").alias("TouchPointCode")
        ).alias("LastUpdateTouchPointSourceSystem"),

        F.lit(None).alias("EformRegFlag"), # 没有对应字段
        
        # RegPersonnelSourceSystem
        F.struct(
            F.col("scon_prs_sourcesystemcode").alias("@Code"),
            F.date_format(F.col("scon_prs_sourcetimestamp"), timestamp_format).alias("SourceTimestamp"),
            F.col("scon_prs_aff_code").alias("AffiliateCode"),
            F.col("scon_prs_mrkt_code").alias("MarketCode"),
            F.col("scon_prs_dvsn_code").alias("DivisionCode"),
            F.col("scon_prs_brnd_code").alias("BrandCode"),
            F.col("scon_registration_prsn_code").alias("PersonnelCode")
        ).alias("RegPersonnelSourceSystem"),
        
        F.col("scon_cntr_isoalpha3code").alias("ConsumerCountryCode_ISO3"),
        F.col("scon_slng_code").alias("SpokenLanguageCode"),
        F.col("scon_wlng_code").alias("WrittenLanguageCode"),
        
        # Personal Info
        F.col("scon_birthmonth").alias("BirthMonth"),
        F.col("scon_birthyear").alias("BirthYear"),
        F.when(F.col("namefilledflag") == 1, True).otherwise(False).alias("NameFilledFlag")
    )

def build_program_list_json_struct():
    """构建ProgramList JSON结构，并将积分/配额小数字段截断为整数"""
    program_list = F.from_json(F.col("ProgramJSON"), program_list_schema)
    program_items = F.transform(
        program_list.getField("Program"),
        lambda p: F.struct(
            p["ApplicationTouchPointCode"].alias("ApplicationTouchPointCode"),
            p["ConsumerGroup"].alias("ConsumerGroup"),
            p["ProgramTypeCode"].alias("ProgramTypeCode"),
            p["ProgramTypeDescription"].alias("ProgramTypeDescription"),
            p["ProgramLevelCode"].alias("ProgramLevelCode"),
            p["ProgramLevelDescription"].alias("ProgramLevelDescription"),
            p["ProgramSystemIDCode"].alias("ProgramSystemIDCode"),
            p["ProgramSystemIDDescription"].alias("ProgramSystemIDDescription"),
            p["MembershipNum"].alias("MembershipNum"),
            p["CardNum"].alias("CardNum"),
            p["StartTimestamp"].alias("StartTimestamp"),
            p["EndTimestamp"].alias("EndTimestamp"),
            p["PointsAcquired"].cast(LongType()).alias("PointsAcquired"),
            p["PointsRedeemed"].cast(LongType()).alias("PointsRedeemed"),
            p["InitialQuota"].cast(LongType()).alias("InitialQuota"),
            p["AvailableQuota"].cast(LongType()).alias("AvailableQuota")
        )
    )
    return F.when(
        program_list.getField("Program").isNull(),
        F.lit(None)
    ).otherwise(
        F.struct(program_items.alias("Program"))
    )


def build_header_json():
    return F.to_json(
                F.struct(
                    F.col("header_action").alias("@Action"),
                    F.expr("uuid()").alias("DocumentUUID"),
                    F.date_format(F.current_timestamp(), timestamp_format).alias("DocumentTimestamp")
                ), 
                options={"ignoreNullFields": "false"}
            )

def remove_null_reg_personnel_code(json_col):
    """删除 RegPersonnelSourceSystem 下为 null 的 @Code 字段。"""
    return F.regexp_replace(
        json_col,
        r'("RegPersonnelSourceSystem":\{)"@Code":null,',
        r'$1'
    )

def remove_null_assigned_personnel_code(json_col):
    """删除 AssignedPersonnelSourceSystem 下为 null 的 @Code 字段。"""
    return F.regexp_replace(
        json_col,
        r'("AssignedPersonnelSourceSystem":\{)"@Code":null,',
        r'$1'
    )

In [0]:
def generate_cbr_processor(task_id):
    # 1. header
    # header_json = (
    #     spark.table(master_consumer_table_name)
    #         .filter(F.col("task_id") == task_id)
    #         .groupBy("SCON_MRKT_CODE", "consumermdmkey")
    #         .agg(
    #             F.when(
    #                 F.count(F.when(F.col("scon_srcc_action") == "DELETE", 1)) > 0, "DELETE"
    #             ).otherwise("CREATE").alias("srcc_action"))
    #         .orderBy("SCON_MRKT_CODE", "consumermdmkey")
    #         .select(
    #             F.col("SCON_MRKT_CODE").alias("MarketCode"),
    #             F.col("consumermdmkey").alias("MDMKey"),
    #             F.to_json(
    #                 F.struct(
    #                     F.col("srcc_action").alias("@Action"),
    #                     F.expr("uuid()").alias("DocumentUUID"),
    #                     F.date_format(F.current_timestamp(), timestamp_format).alias("DocumentTimestamp")
    #                 ), 
    #                 options={"ignoreNullFields": "false"}
    #             ).alias("Header_JSON")
    #         )
    #     .select("MarketCode", "MDMKey", "Header_JSON")
    # )

    # 1.1 获取ukey
    consumermdmkey_df = (spark.table(master_derived_consumer_l1_table_name)
        .filter(F.col("task_id") == task_id)
        .select("scon_mrkt_code", "consumermdmkey"))
    
    consumermdmkey_df.cache()
    print(f"ukey count: {consumermdmkey_df.count()}")

    # 1.2 获取KOR DJ ukey. cbrl表里排除KOR DJ Brand数据，KOR DJ Brand单独存一张cbrldrjart表
    brand43_keys = (spark.table(master_derived_consumer_l2_table_name)
            .filter(F.col("task_id") == task_id)
            .filter((F.col("SCON_MRKT_CODE") == "KOR") & (F.col("scon_brnd_code") == "43"))
            .select(
               F.col("SCON_MRKT_CODE").alias("MarketCode"), 
               F.col("consumermdmkey").alias("MDMKey")
            ))      


    # 读取L2表
    df_l2 = (
        spark.table(master_derived_consumer_l2_table_name).alias("l2_tab")
            # .filter(F.col("task_id") == task_id)
            .join(consumermdmkey_df.alias("ukey_tab"), ["scon_mrkt_code", "consumermdmkey"], "inner")
            .select(
                "l2_tab.consumermdmkey",
                "scvlevel",
                F.lit(None).alias("tcpm_distributionchannelcode"),  # 空字符串
                "scon_srcs_code",
                "scon_sourcetimestamp",
                "l2_tab.scon_mrkt_code",
                "scon_aff_code",
                "scon_dvsn_code",
                "scon_brnd_code",
                "scon_salutation",
                "scon_englishfirstname",
                "scon_englishmiddlename",
                "scon_englishlastname",
                "scon_englishfullname",
                "scon_localfirstname",
                "scon_localmiddlename",
                "scon_locallastname",
                "scon_localfullname",
                "scon_localfirstname2",
                "scon_localmiddlename2",
                "scon_locallastname2",
                "scon_localfullname2",
                "scon_gndr_code",
                "scon_birthday",
                "scon_birthmonth",
                "scon_birthyear",
                "scon_identitynum",
                "scon_passportnum",
                "scon_socialsecuritynum",
                "scon_clas_code",
                "scon_reg_dt",
                "scon_reg_sourcetimestamp",
                "scon_registration_toch_code",
                "scon_prsn_srcs_code",
                "scon_prsn_sourcetimestamp",
                "scon_prsn_aff_code",
                "scon_prsn_mrkt_code",
                "scon_prsn_dvsn_code",
                "scon_prsn_brnd_code",
                "scon_registration_prsn_code",
                "scon_last_update_timestamp",
                "scon_last_update_toch_code",
                "scon_pre_toch_sourcetimestamp",
                "scon_pre_toch_aff_code",
                "scon_pre_toch_mrkt_code",
                "scon_pre_toch_dvsn_code",
                "scon_pre_toch_brnd_code",
                "scon_preferred_toch_code",
                "scon_assn_prsn_srcs_code",
                "scon_assn_prsn_sourcetimestamp",
                "scon_assn_prsn_aff_code",
                "scon_assn_prsn_mrkt_code",
                "scon_assn_prsn_dvsn_code",
                "scon_assn_prsn_brnd_code",
                "scon_assigned_prsn_code",
                "scon_wlng_code",
                "scon_slng_code",
                "scon_cntr_isoalpha3code",
                "scon_ethn_mrkt_code",
                "scon_ethn_code",
                "scon_sknt_brnd_code",
                "scon_sknt_code",
                "scon_hairt_code",
                "scon_cvls_code",
                "scon_company",
                "scon_department",
                "scon_jobtitle",
                "scon_yearlyincome",
                "scon_donotcontact_flag",
                "scon_curr_code",
                "scon_agefrom",
                "scon_ageto",
                "scon_nationality",
                "scon_channel",
                "scon_preferred_comm_channel",
                "scon_anniversary_dt",
                "scon_commercial_flag",
                "scon_emailreceipt_flag",
                "scon_prospect_flag",
                "scon_active_flag",
                "scon_hrrequesttimestamp",
                "scon_status",
                F.when(F.col("namefilledflag") == 1, True).otherwise(False).alias("namefilledflag"),
                "RecordTimeStamp",
                "DervivedBestRecordListID",
                "SourceSystemJSON",
                "HobbyJSON",
                "EMediaJSON",
                "PhoneJSON",
                "AddressJSON",
                "OptInJSON",
                "CrossBrandOptInJSON",
                "CustomAttributesJSON",
                "AuxiliaryAttributesJSON",
                "HairTypeJSON",
                "MakeUpConcernsJSON",
                "HairConcernsJSON",
                "SkinConcernsJSON",
                "TermsJSON",
                "ConsumerGroupJSON",
                "RemarkJSON",
                "NotesJSON",
                "scon_firstpurchasedate"
            )
    )

    # 读取L3表
    df_l3 = (
        spark.table(master_derived_consumer_l3_table_name).alias("l3_tab")
            # .filter(F.col("task_id") == task_id)
            .join(consumermdmkey_df.alias("ukey_tab"), ["scon_mrkt_code", "consumermdmkey"], "inner")
            .select(
                "l3_tab.consumermdmkey",
                "scvlevel",
                "tcpm_distributionchannelcode",  # L3 有真实值
                "scon_srcs_code",
                "scon_sourcetimestamp",
                "l3_tab.scon_mrkt_code",
                "scon_aff_code",
                "scon_dvsn_code",
                "scon_brnd_code",
                "scon_salutation",
                "scon_englishfirstname",
                "scon_englishmiddlename",
                "scon_englishlastname",
                "scon_englishfullname",
                "scon_localfirstname",
                "scon_localmiddlename",
                "scon_locallastname",
                "scon_localfullname",
                "scon_localfirstname2",
                "scon_localmiddlename2",
                "scon_locallastname2",
                "scon_localfullname2",
                "scon_gndr_code",
                "scon_birthday",
                "scon_birthmonth",
                "scon_birthyear",
                "scon_identitynum",
                "scon_passportnum",
                "scon_socialsecuritynum",
                "scon_clas_code",
                "scon_reg_dt",
                "scon_reg_sourcetimestamp",
                "scon_registration_toch_code",
                "scon_prsn_srcs_code",
                "scon_prsn_sourcetimestamp",
                "scon_prsn_aff_code",
                "scon_prsn_mrkt_code",
                "scon_prsn_dvsn_code",
                "scon_prsn_brnd_code",
                "scon_registration_prsn_code",
                "scon_last_update_timestamp",
                "scon_last_update_toch_code",
                "scon_pre_toch_sourcetimestamp",
                "scon_pre_toch_aff_code",
                "scon_pre_toch_mrkt_code",
                "scon_pre_toch_dvsn_code",
                "scon_pre_toch_brnd_code",
                "scon_preferred_toch_code",
                "scon_assn_prsn_srcs_code",
                "scon_assn_prsn_sourcetimestamp",
                "scon_assn_prsn_aff_code",
                "scon_assn_prsn_mrkt_code",
                "scon_assn_prsn_dvsn_code",
                "scon_assn_prsn_brnd_code",
                "scon_assigned_prsn_code",
                "scon_wlng_code",
                "scon_slng_code",
                "scon_cntr_isoalpha3code",
                "scon_ethn_mrkt_code",
                "scon_ethn_code",
                "scon_sknt_brnd_code",
                "scon_sknt_code",
                "scon_hairt_code",
                "scon_cvls_code",
                "scon_company",
                "scon_department",
                "scon_jobtitle",
                "scon_yearlyincome",
                "scon_donotcontact_flag",
                "scon_curr_code",
                "scon_agefrom",
                "scon_ageto",
                "scon_nationality",
                "scon_channel",
                "scon_preferred_comm_channel",
                "scon_anniversary_dt",
                "scon_commercial_flag",
                "scon_emailreceipt_flag",
                "scon_prospect_flag",
                "scon_active_flag",
                "scon_hrrequesttimestamp",
                "scon_status",
                F.when(F.col("namefilledflag") == 1, True).otherwise(False).alias("namefilledflag"),
                "RecordTimeStamp",
                "DervivedBestRecordListID",
                "SourceSystemJSON",
                "HobbyJSON",
                "EMediaJSON",
                "PhoneJSON",
                "AddressJSON",
                "OptInJSON",
                "CrossBrandOptInJSON",
                "CustomAttributesJSON",
                "AuxiliaryAttributesJSON",
                "HairTypeJSON",
                "MakeUpConcernsJSON",
                "HairConcernsJSON",
                "SkinConcernsJSON",
                "TermsJSON",
                "ConsumerGroupJSON",
                "RemarkJSON",
                "NotesJSON",
                "scon_firstpurchasedate"
            )
    )

    # 合并L2 & L3
    query_union_l2_l3 = df_l2.unionByName(df_l3)

    dbr_data = (
        query_union_l2_l3
        .select(
            F.col("scon_mrkt_code"),
            F.col("consumermdmkey"),
            F.col("scvlevel").alias("@Level"),
            F.from_json(F.col("SourceSystemJSON"), source_system_list_schema).alias("SourceSystemList"),
            # 构建Attributes
            build_attributes_json_struct().alias("Attributes"),
            # 构建PersonalData
            build_personal_data_json_struct().alias("PersonalData"),
            # 构建NonPii PersonalData
            build_nonpii_personal_data_json_struct().alias("PersonalData_NonPii"),
            # 构建ContactInformation
            build_contact_information_json().alias("ContactInformation"),
            # 构建NonPii ContactInformation
            build_nonpii_contact_information_json().alias("ContactInformation_NonPii"),
            F.from_json(F.col("OptInJSON"), opt_in_list_schema).alias("OptInList"),
            F.from_json(F.col("CrossBrandOptInJSON"), cross_brand_opt_in_list_schema).alias("CrossBrandOptInList"),
            F.from_json(F.col("AuxiliaryAttributesJSON"), auxiliary_attribute_list_schema).alias("AuxiliaryAttributeList"),
            F.from_json(F.col("TermsJSON"), terms_and_condition_list_schema).alias("TermsAndConditionList"),
            F.from_json(F.col("CustomAttributesJSON"), custom_attribute_list_schema).alias("CustomAttributeList"),
            F.from_json(F.col("ConsumerGroupJSON"), customer_group_list_schema).alias("CustomerGroupList"),
            F.from_json(F.col("RemarkJSON"), remark_list_schema).alias("RemarkList"),
            F.from_json(F.col("NotesJSON"), note_list_schema).alias("NoteList")
        )
        .cache()
    )

    # 2.1 DBR
    # 生成 DBR JSON
    dbr_json = (
        dbr_data
        .withColumn("dbr", F.struct(
            F.col("@Level"),
            F.col("SourceSystemList"),
            F.col("Attributes"),
            F.col("PersonalData"),
            F.col("ContactInformation"),
            F.col("OptInList"),
            F.col("CrossBrandOptInList"),
            F.col("AuxiliaryAttributeList"),
            F.col("TermsAndConditionList"),
            F.col("CustomAttributeList"),
            F.col("CustomerGroupList"),
            F.col("RemarkList"),
            F.col("NoteList")
        ))
        .select("scon_mrkt_code", "consumermdmkey", "dbr")
        .distinct()
        .groupBy("scon_mrkt_code", "consumermdmkey")
        .agg(
            F.struct(
                F.sort_array(F.collect_list(F.col("dbr")), asc=True).alias("DerivedBestRecord")
            ).alias("dbr_list")
        )
        .orderBy("scon_mrkt_code", "consumermdmkey")
        .select(
            F.col("scon_mrkt_code").alias("MarketCode"),
            F.col("consumermdmkey").alias("MDMKey"),
            F.to_json(F.col("dbr_list"), options={"ignoreNullFields": "false"}).alias("DBR_Merged")
        )
        .withColumn(
            "DBR_Merged",
            remove_null_assigned_personnel_code(
                remove_null_reg_personnel_code(F.col("DBR_Merged"))
            )
        )
    )

    # 2.2 NonPii DBR
    # 生成 NonPii DBR JSON
    dbr_nonpii_json = (
        dbr_data
        .withColumn("dbr", F.struct(
            F.col("@Level"),
            F.col("SourceSystemList"),
            F.col("Attributes"),
            F.col("PersonalData_NonPii").alias("PersonalData"),
            F.col("ContactInformation_NonPii").alias("ContactInformation"),
            F.col("OptInList"),
            F.col("CrossBrandOptInList"),
            F.col("AuxiliaryAttributeList"),
            F.col("TermsAndConditionList"),
            F.col("CustomAttributeList"),
            F.col("CustomerGroupList"),
            F.col("RemarkList"),
            F.col("NoteList")
        ))
        .select("scon_mrkt_code", "consumermdmkey", "dbr")
        .distinct()
        .groupBy("scon_mrkt_code", "consumermdmkey")
        .agg(
            F.struct(
                F.sort_array(F.collect_list(F.col("dbr")), asc=True).alias("DerivedBestRecord")
            ).alias("dbr_list")
        )
        .orderBy("scon_mrkt_code", "consumermdmkey")
        .select(
            F.col("scon_mrkt_code").alias("MarketCode"),
            F.col("consumermdmkey").alias("MDMKey"),
            F.to_json(F.col("dbr_list"), options={"ignoreNullFields": "false"}).alias("DBR_Merged")
        )
        .withColumn(
            "DBR_Merged",
            remove_null_assigned_personnel_code(
                remove_null_reg_personnel_code(F.col("DBR_Merged"))
            )
        )
    )

    # 3.1 cbr
    # cbr data
    cbr_data = (
        spark.table(master_derived_consumer_l1_table_name)
        .filter(F.col("task_id") == task_id)
        .select(
            "consumermdmkey",
            "scon_aff_code",
            "scon_mrkt_code",
            "scon_dvsn_code",
            "scon_brnd_code",
            "scon_srcs_code",
            "scon_sourcetimestamp",
            "scon_gndr_code",
            "scon_birthday",
            "scon_birthmonth",
            "scon_birthyear",
            "scon_reg_dt",
            "scon_reg_sourcetimestamp",
            "scon_registration_toch_code",
            "scon_prs_sourcesystemcode",
            "scon_prs_sourcetimestamp",
            "scon_prs_aff_code",
            "scon_prs_mrkt_code",
            "scon_prs_dvsn_code",
            "scon_prs_brnd_code",
            "scon_registration_prsn_code",
            "scon_lastupdate_toch_code",
            "scon_wlng_code",
            "scon_slng_code",
            "scon_cntr_isoalpha3code",
            "scon_cvls_code",
            "namefilledflag",
            "RecordTimeStamp",
            "RecordUUID",
            "SourceSystemJSON",
            "CustomAttributesJSON",
            "ProgramJSON",
            "header_action"
        )
        .distinct()
        .select(
            F.col("scon_mrkt_code").alias("MarketCode"),
            F.col("consumermdmkey").alias("MDMKey"),
            F.from_json(F.col("SourceSystemJSON"), source_system_list_schema).alias("SourceSystemList"),
            build_cbr_attributes_json_struct().alias("Attributes"),
            build_cbr_personal_data_json_struct().alias("PersonalData"),
            build_nonpii_cbr_personal_data_json_struct().alias("PersonalData_NonPii"),
            build_program_list_json_struct().alias("ProgramList"),
            F.from_json(F.col("CustomAttributesJSON"), custom_attribute_list_schema).alias("CustomAttributeList"),

            build_header_json().alias("Header_JSON")
        )
        # .orderBy("MarketCode", "MDMKey")
    )

    # 生成 CBR JSON
    cbr_json = (
        cbr_data
        .select(
            F.col("MarketCode"),
            F.col("MDMKey"),
            F.col("Header_JSON"),
            F.to_json(
                F.struct(
                    F.col("SourceSystemList"),
                    F.col("Attributes"),
                    F.col("PersonalData"),
                    F.col("ProgramList"),
                    F.col("CustomAttributeList")
                ),
                options={"ignoreNullFields": "false"}
            ).alias("BR_JSON")
        )
        .withColumn(
            "BR_JSON", remove_null_reg_personnel_code(F.col("BR_JSON"))
        )
    )

    # 3.2 NonPii CBR
    # 生成 NonPii CBR JSON
    cbr_nonpii_json = (
        cbr_data
        .select(
            F.col("MarketCode"),
            F.col("MDMKey"),
            F.col("Header_JSON"),
            F.to_json(
                F.struct(
                    F.col("SourceSystemList"),
                    F.col("Attributes"),
                    F.col("PersonalData_NonPii").alias("PersonalData"),
                    F.col("ProgramList"),
                    F.col("CustomAttributeList")
                ),
                options={"ignoreNullFields": "false"}
            ).alias("BR_JSON")
        )
        .withColumn(
            "BR_JSON", remove_null_reg_personnel_code(F.col("BR_JSON"))
        )
    )

    # 4.1 final CBR
    # 生成 CBR JSON
    final_json = (
        cbr_json
        .join(dbr_json, on=["MarketCode", "MDMKey"], how="inner")
        # .join(header_json, on=["MarketCode", "MDMKey"], how="inner")
        .select(
            "MarketCode",
            "MDMKey",
            F.concat(
                F.lit("{\"Header\":"),
                F.col("Header_JSON"),
                F.lit(",\"ConsumerBestRecord\": {\"@RecordUUID\": \""),
                F.expr("uuid()"),
                F.lit("\",\"BestRecord\":"),
                F.col("BR_JSON"),
                F.lit(",\"DerivedBestRecordList\":"),
                F.col("DBR_Merged"),
                F.lit("}}")
            ).alias("FinalJSON"),
            F.current_timestamp().alias("CreatedTimestamp"),
            F.current_timestamp().alias("UpdatedTimestamp"),
            F.lit(task_id).alias("TASK_ID")
        )
        # 移除指定空数组字段 + 替换其余空数组为null
        .withColumn("FinalJSON", F.regexp_replace(F.regexp_replace(F.col("FinalJSON"), pattern, ""), r'\[\]', 'null'))
    )

    # 4.2 final NonPii CBR
    # 生成 NonPii CBR JSON
    nonpii_final_json = (
        cbr_nonpii_json
        .join(dbr_nonpii_json, on=["MarketCode", "MDMKey"], how="inner")
        # .join(header_json, on=["MarketCode", "MDMKey"], how="inner")
        .select(
            "MarketCode",
            "MDMKey",
            F.concat(
                F.lit("{\"Header\":"),
                F.col("Header_JSON"),
                F.lit(",\"ConsumerBestRecord\": {\"@RecordUUID\": \""),
                F.expr("uuid()"),
                F.lit("\",\"BestRecord\":"),
                F.col("BR_JSON"),
                F.lit(",\"DerivedBestRecordList\":"),
                F.col("DBR_Merged"),
                F.lit("}}")
            ).alias("FinalJSON"),
            F.current_timestamp().alias("CreatedTimestamp"),
            F.current_timestamp().alias("UpdatedTimestamp"),
            F.lit(task_id).alias("TASK_ID")
        )
        # 移除指定空数组字段 + 替换其余空数组为null
        .withColumn("FinalJSON", F.regexp_replace(F.regexp_replace(F.col("FinalJSON"), pattern, ""), r'\[\]', 'null'))
    )

    # 5. DJ cbr
    # cbrl表里排除KOR DJ Brand数据，KOR DJ Brand单独存一张cbrldrjart表
    # brand43_keys = (spark.table(master_consumer_table_name).filter(F.col("task_id") == task_id)
    #              .filter((F.col("SCON_MRKT_CODE") == "KOR") & (F.col("scon_brnd_code") == "43"))
    #              .select(F.col("SCON_MRKT_CODE").alias("MarketCode"), 
    #                      F.col("consumermdmkey").alias("MDMKey"))
    #              .distinct())
    
    final_json_exclude_brand43 = final_json.join(brand43_keys, ["MarketCode","MDMKey"], "left_anti")
    final_json_brand43 = final_json.join(brand43_keys, ["MarketCode","MDMKey"], "inner")

    final_json_exclude_brand43.cache()
    final_json_brand43.cache()
    nonpii_final_json.cache()

    final_json_exclude_brand43_count = max(final_json_exclude_brand43.count(), 0)
    # 插入 CBR 数据到目标表（不含KOR DJ）
    if final_json_exclude_brand43_count > 0:
        merge_condition = get_merge_condition(["MarketCode", "MDMKey"])
        merge_t_table(dataset_cbr_table_name, final_json_exclude_brand43, merge_condition)
        # calc ctable
        calc_ctable(dataset_cbr_table_name, None)

    final_json_brand43_count = max(final_json_brand43.count(), 0)
    # 插入 CBR 数据到目标表（只有KOR DJ）
    if final_json_brand43_count > 0:
        merge_condition = get_merge_condition(["MarketCode", "MDMKey"])
        merge_t_table(dataset_cbrldrjart_table_name, final_json_brand43, merge_condition)
        # calc ctable
        calc_ctable(dataset_cbrldrjart_table_name, None)

    nonpii_final_json_count = max(nonpii_final_json.count(), 0)
    # 插入 NonPii CBR 数据到目标表
    if nonpii_final_json_count > 0:
        merge_condition = get_merge_condition(["MarketCode", "MDMKey"])
        merge_t_table(dataset_nonpii_cbr_table_name, nonpii_final_json, merge_condition)
        # calc ctable
        calc_ctable(dataset_nonpii_cbr_table_name, None)

    dbr_data.unpersist()
    
    final_json_exclude_brand43.unpersist()
    final_json_brand43.unpersist()
    nonpii_final_json.unpersist()

In [0]:
with StepLogger("generate_cbr_dataset", "07", "consumerlist", task_id=task_id) as logger:
    generate_cbr_processor(task_id)